# Nepali Grammar Checker — PRODUCTION INFERENCE (BULLETPROOF)

**Handles:**
- Corrupted checkpoint files
- Vocab size mismatches
- Missing tokenizer files
- Automatic model reconstruction


In [23]:
import torch
import torch.nn as nn
import torch.optim as optim
import json
import os
import math
from typing import List, Tuple, Dict, Optional
import warnings
warnings.filterwarnings('ignore')

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")

PyTorch: 2.12.0+cu130
CUDA: True


## Step 1: Define all models & tokenizers


In [24]:
# ============================================================================
# CHARACTER TOKENIZER
# ============================================================================

class CharTokenizer:
    PAD, SOS, EOS, UNK = "<PAD>", "<SOS>", "<EOS>", "<UNK>"

    def __init__(self):
        self.char2idx = {}
        self.idx2char = {}
        self.vocab_size = 0

    def build_vocab(self, texts):
        chars = set()
        for t in texts:
            chars.update(list(str(t)))
        specials = [self.PAD, self.SOS, self.EOS, self.UNK]
        all_chars = specials + sorted(chars)
        self.char2idx = {c: i for i, c in enumerate(all_chars)}
        self.idx2char = {i: c for c, i in self.char2idx.items()}
        self.vocab_size = len(self.char2idx)

    def encode(self, text, max_len=30, add_sos=False, add_eos=False):
        ids = []
        if add_sos:
            ids.append(self.char2idx[self.SOS])
        for c in str(text):
            ids.append(self.char2idx.get(c, self.char2idx[self.UNK]))
        if add_eos:
            ids.append(self.char2idx[self.EOS])
        ids = ids[:max_len]
        ids += [self.char2idx[self.PAD]] * (max_len - len(ids))
        return ids

    def decode(self, ids):
        out = []
        for i in ids:
            c = self.idx2char.get(i, self.UNK)
            if c in (self.PAD, self.SOS):
                continue
            if c == self.EOS:
                break
            out.append(c)
        return "".join(out)

    def save(self, filepath):
        with open(filepath, 'w', encoding='utf-8') as f:
            json.dump({
                "char2idx": self.char2idx,
                "idx2char": {str(k): v for k, v in self.idx2char.items()}
            }, f, ensure_ascii=False, indent=2)

    @classmethod
    def load(cls, filepath):
        tok = cls()
        with open(filepath, 'r', encoding='utf-8') as f:
            data = json.load(f)
        tok.char2idx = data["char2idx"]
        tok.idx2char = {int(k): v for k, v in data["idx2char"].items()}
        tok.vocab_size = len(tok.char2idx)
        return tok


print("✓ CharTokenizer defined")

✓ CharTokenizer defined


In [25]:
# ============================================================================
# DETECTION MODEL
# ============================================================================

class CharTransformerDetector(nn.Module):
    """Transformer encoder for char-level wrong-word detection"""
    def __init__(self, vocab_size, embed_dim=64, num_heads=4,
                 num_layers=3, ff_dim=256, max_len=30, dropout=0.3):
        super().__init__()
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.pos_embedding = nn.Embedding(max_len, embed_dim)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=ff_dim,
            dropout=dropout,
            batch_first=True,
            norm_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        B, T = x.shape
        positions = torch.arange(T, device=x.device).unsqueeze(0).expand(B, T)
        out = self.dropout(self.embedding(x) + self.pos_embedding(positions))
        pad_mask = (x == 0)
        out = self.transformer(out, src_key_padding_mask=pad_mask)
        mask_f = (~pad_mask).float().unsqueeze(-1)
        pooled = (out * mask_f).sum(dim=1) / mask_f.sum(dim=1).clamp(min=1)
        return self.classifier(pooled).squeeze(-1)


print("✓ CharTransformerDetector defined")

✓ CharTransformerDetector defined


In [26]:
# ============================================================================
# SEQ2SEQ CORRECTION MODEL
# ============================================================================

class TransformerEncoder(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, num_heads=4,
                 num_layers=3, ff_dim=512, max_len=30, dropout=0.1):
        super().__init__()
        self.embed_dim = embed_dim
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.pos_embed = nn.Embedding(max_len + 2, embed_dim)
        
        layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads,
            dim_feedforward=ff_dim, dropout=dropout,
            batch_first=True, norm_first=True
        )
        self.transformer = nn.TransformerEncoder(layer, num_layers=num_layers)
        self.dropout = nn.Dropout(dropout)
        self.fc_h = nn.Linear(embed_dim, embed_dim)
        self.fc_c = nn.Linear(embed_dim, embed_dim)

    def forward(self, x):
        B, T = x.shape
        pos = torch.arange(T, device=x.device).unsqueeze(0).expand(B, T)
        out = self.dropout(self.embedding(x) + self.pos_embed(pos))
        pad_mask = (x == 0)
        enc_out = self.transformer(out, src_key_padding_mask=pad_mask)
        mask_f = (~pad_mask).float().unsqueeze(-1)
        mean_enc = (enc_out * mask_f).sum(1) / mask_f.sum(1).clamp(min=1)
        h = torch.tanh(self.fc_h(mean_enc)).unsqueeze(0)
        c = torch.tanh(self.fc_c(mean_enc)).unsqueeze(0)
        return enc_out, h, c


class BahdanauAttention(nn.Module):
    def __init__(self, hidden_dim, encoder_dim):
        super().__init__()
        self.W1 = nn.Linear(encoder_dim, hidden_dim)
        self.W2 = nn.Linear(hidden_dim, hidden_dim)
        self.v = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, dec_h, enc_out):
        score = self.v(torch.tanh(
            self.W1(enc_out) + self.W2(dec_h).unsqueeze(1)
        )).squeeze(-1)
        weights = torch.softmax(score, dim=1)
        context = torch.bmm(weights.unsqueeze(1), enc_out).squeeze(1)
        return context, weights


class LSTMDecoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, encoder_dim, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.attention = BahdanauAttention(hidden_dim, encoder_dim)
        self.lstm = nn.LSTMCell(embed_dim + encoder_dim, hidden_dim)
        self.fc_out = nn.Linear(hidden_dim + encoder_dim + embed_dim, vocab_size)
        self.dropout = nn.Dropout(dropout)

    def forward_step(self, token, h, c, enc_out):
        emb = self.dropout(self.embedding(token))
        context, weights = self.attention(h, enc_out)
        h, c = self.lstm(torch.cat([emb, context], dim=1), (h, c))
        pred = self.fc_out(torch.cat([h, context, emb], dim=1))
        return pred, h, c, weights


class Seq2SeqCorrector(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=128,
                 enc_layers=3, max_word_len=20, dropout=0.1):
        super().__init__()
        self.vocab_size = vocab_size
        self.encoder = TransformerEncoder(
            vocab_size, embed_dim, num_heads=4,
            num_layers=enc_layers, ff_dim=512,
            max_len=max_word_len, dropout=dropout
        )
        self.decoder = LSTMDecoder(
            vocab_size, embed_dim, hidden_dim,
            encoder_dim=embed_dim, dropout=dropout
        )

    def forward(self, src, tgt, teacher_forcing_ratio=0.5):
        B, tgt_len = tgt.shape
        enc_out, h, c = self.encoder(src)
        h, c = h.squeeze(0), c.squeeze(0)
        input_tok = tgt[:, 0]
        outputs = torch.zeros(B, tgt_len, self.vocab_size).to(src.device)
        for t in range(1, tgt_len):
            pred, h, c, _ = self.decoder.forward_step(input_tok, h, c, enc_out)
            outputs[:, t] = pred
            use_teacher = torch.rand(1).item() < teacher_forcing_ratio
            input_tok = tgt[:, t] if use_teacher else pred.argmax(dim=1)
        return outputs


print("✓ Seq2SeqCorrector defined")

✓ Seq2SeqCorrector defined


## Step 2: Load tokenizers from files

In [27]:
# ============================================================================
# LOAD TOKENIZERS (with fallback to training data)
# ============================================================================

def load_tokenizers():
    """
    Load saved tokenizers. If missing, use training data to rebuild.
    """
    detect_tok = None
    seq2seq_tok = None
    
    # Try to load detector tokenizer
    if os.path.exists("detect_char_tokenizer.json"):
        try:
            detect_tok = CharTokenizer.load("detect_char_tokenizer.json")
            print(f"✓ Detector tokenizer loaded (vocab: {detect_tok.vocab_size})")
        except Exception as e:
            print(f"⚠️  Failed to load detector tokenizer: {e}")
    
    # Try to load seq2seq tokenizer
    if os.path.exists("seq2seq_char_tokenizer.json"):
        try:
            seq2seq_tok = CharTokenizer.load("seq2seq_char_tokenizer.json")
            print(f"✓ Seq2Seq tokenizer loaded (vocab: {seq2seq_tok.vocab_size})")
        except Exception as e:
            print(f"⚠️  Failed to load seq2seq tokenizer: {e}")
    
    # Fallback: Create dummy tokenizers with Nepali characters
    if detect_tok is None:
        print("\n⚠️  Creating fallback detector tokenizer...")
        detect_tok = CharTokenizer()
        nepali_chars = list("अआइईउऊऋएऐओऔकखगघङचछजझञटठडढणतथदधनपफबभमयरलवशषसहक्षत्रज्ञाइीुूृेैोौंः:ँ०१२३४५६७८९")
        detect_tok.build_vocab(nepali_chars + ["a", "b", "c", "d", "e", "f"])
        print(f"  Created fallback (vocab: {detect_tok.vocab_size})")
    
    if seq2seq_tok is None:
        print("\n⚠️  Creating fallback seq2seq tokenizer...")
        seq2seq_tok = CharTokenizer()
        nepali_chars = list("अआइईउऊऋएऐओऔकखगघङचछजझञटठडढणतथदधनपफबभमयरलवशषसहक्षत्रज्ञाइीुूृेैोौंः:ँ०१२३४५६७८९")
        seq2seq_tok.build_vocab(nepali_chars + ["a", "b", "c", "d", "e", "f"])
        print(f"  Created fallback (vocab: {seq2seq_tok.vocab_size})")
    
    return detect_tok, seq2seq_tok


detect_tok, seq2seq_tok = load_tokenizers()
print(f"\nDetector vocab size: {detect_tok.vocab_size}")
print(f"Seq2Seq vocab size: {seq2seq_tok.vocab_size}")

✓ Detector tokenizer loaded (vocab: 74)
✓ Seq2Seq tokenizer loaded (vocab: 82)

Detector vocab size: 74
Seq2Seq vocab size: 82


## Step 3: Load checkpoints with automatic vocab size handling

In [28]:
# ============================================================================
# CHECKPOINT LOADER (Bulletproof)
# ============================================================================

def safe_load_checkpoint(filepath, device='cpu'):
    """
    Load checkpoint and inspect it.
    Returns: checkpoint dict or None if corrupted
    """
    if not os.path.exists(filepath):
        print(f"❌ File not found: {filepath}")
        return None
    
    try:
        print(f"Loading {os.path.basename(filepath)}...")
        checkpoint = torch.load(filepath, map_location=device, weights_only=False)
        print(f"✓ Loaded successfully")
        
        # If wrapped in a dict, extract state_dict
        if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
            return checkpoint["model_state_dict"]
        return checkpoint
    except Exception as e:
        print(f"❌ Failed to load: {str(e)[:100]}")
        return None


def get_checkpoint_vocab_size(state_dict):
    """
    Inspect checkpoint and extract vocab size from embedding layers.
    """
    if state_dict is None:
        return None
    
    try:
        # Check for embedding weight shape
        if "embedding.weight" in state_dict:
            vocab_size = state_dict["embedding.weight"].shape[0]
            return vocab_size
        return None
    except:
        return None


# Load checkpoints
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}\n")

detector_ckpt = safe_load_checkpoint("detector_best.pth", device=device)
seq2seq_ckpt = safe_load_checkpoint("seq2seq_best copy.pth", device=device)

if detector_ckpt:
    det_vocab = get_checkpoint_vocab_size(detector_ckpt)
    print(f"  Detector checkpoint vocab: {det_vocab}\n")

if seq2seq_ckpt:
    seq2seq_vocab = get_checkpoint_vocab_size(seq2seq_ckpt)
    print(f"  Seq2Seq checkpoint vocab: {seq2seq_vocab}\n")

Using device: cuda

Loading detector_best.pth...
✓ Loaded successfully
Loading seq2seq_best copy.pth...
✓ Loaded successfully
  Detector checkpoint vocab: 74

  Seq2Seq checkpoint vocab: None



In [29]:
# ============================================================================
# CREATE MODELS WITH CORRECT VOCAB SIZES & LOAD CHECKPOINTS
# ============================================================================

MAX_SEQ_LEN = 30
MAX_WORD_LEN = 20

# Create detector with ACTUAL tokenizer vocab size
detector = CharTransformerDetector(
    vocab_size=detect_tok.vocab_size,
    embed_dim=64,
    num_heads=4,
    num_layers=3,
    ff_dim=256,
    max_len=MAX_SEQ_LEN,
    dropout=0.3
).to(device)

if detector_ckpt is not None:
    try:
        # Load only matching weights (skip mismatched ones)
        incompatible = detector.load_state_dict(detector_ckpt, strict=False)
        if incompatible.missing_keys:
            print(f"⚠️  Detector: {len(incompatible.missing_keys)} missing keys")
        if incompatible.unexpected_keys:
            print(f"⚠️  Detector: {len(incompatible.unexpected_keys)} unexpected keys")
        print(f"✓ Detector checkpoint loaded (non-strict)\n")
    except Exception as e:
        print(f"❌ Detector load failed: {e}\n")
else:
    print(f"⚠️  No detector checkpoint, using random init\n")

detector.eval()
print(f"Detector: {sum(p.numel() for p in detector.parameters()):,} params")

# Create seq2seq with ACTUAL tokenizer vocab size
s2s_model = Seq2SeqCorrector(
    vocab_size=seq2seq_tok.vocab_size,
    embed_dim=128,
    hidden_dim=128,
    enc_layers=3,
    max_word_len=MAX_WORD_LEN,
    dropout=0.1
).to(device)

if seq2seq_ckpt is not None:
    try:
        # Load only matching weights (skip mismatched ones)
        incompatible = s2s_model.load_state_dict(seq2seq_ckpt, strict=False)
        if incompatible.missing_keys:
            print(f"⚠️  Seq2Seq: {len(incompatible.missing_keys)} missing keys")
        if incompatible.unexpected_keys:
            print(f"⚠️  Seq2Seq: {len(incompatible.unexpected_keys)} unexpected keys")
        print(f"✓ Seq2Seq checkpoint loaded (non-strict)\n")
    except Exception as e:
        print(f"❌ Seq2Seq load failed: {e}\n")
else:
    print(f"⚠️  No seq2seq checkpoint, using random init\n")

s2s_model.eval()
print(f"Seq2Seq: {sum(p.numel() for p in s2s_model.parameters()):,} params")

✓ Detector checkpoint loaded (non-strict)

Detector: 160,833 params
✓ Seq2Seq checkpoint loaded (non-strict)

Seq2Seq: 914,002 params


## Step 4: Inference functions

In [30]:
# ============================================================================
# INFERENCE FUNCTIONS
# ============================================================================

def detect_word(word, threshold=0.5):
    """
    Detect if a word is correct.
    Returns: (is_correct: bool, confidence: float)
    """
    detector.eval()
    indices = detect_tok.encode(word, MAX_SEQ_LEN)
    x = torch.tensor([indices], dtype=torch.long).to(device)
    
    with torch.no_grad():
        prob = detector(x).item()
    
    is_correct = prob > threshold
    return is_correct, round(prob, 4)


def correct_word_beam(wrong_word, beam_width=5, max_len=20):
    """
    Beam search for top-3 corrections.
    Returns: [(word, confidence), ...]
    """
    s2s_model.eval()
    src = torch.tensor(
        [seq2seq_tok.encode(str(wrong_word), max_len)], dtype=torch.long
    ).to(device)

    SOS = seq2seq_tok.char2idx[seq2seq_tok.SOS]
    EOS = seq2seq_tok.char2idx[seq2seq_tok.EOS]
    PAD = seq2seq_tok.char2idx[seq2seq_tok.PAD]

    with torch.no_grad():
        enc_out, h, c = s2s_model.encoder(src)
    h = h.squeeze(0)
    c = c.squeeze(0)

    beams = [(0.0, [], h, c)]
    completed = []

    for _ in range(max_len):
        if not beams:
            break
        candidates = []
        for lp, tokens, bh, bc in beams:
            if tokens and tokens[-1] == EOS:
                completed.append((lp, tokens))
                continue
            last = torch.tensor(
                [tokens[-1] if tokens else SOS], dtype=torch.long
            ).to(device)
            with torch.no_grad():
                pred, new_h, new_c, _ = s2s_model.decoder.forward_step(
                    last, bh, bc, enc_out
                )
            log_p = torch.log_softmax(pred[0], dim=-1)
            topk_lp, topk_idx = log_p.topk(beam_width)
            for tlp, tidx in zip(topk_lp.tolist(), topk_idx.tolist()):
                candidates.append((lp + tlp, tokens + [tidx], new_h, new_c))
        candidates.sort(key=lambda x: x[0], reverse=True)
        beams = candidates[:beam_width]

    for lp, tokens, _, _ in beams:
        completed.append((lp, tokens))
    completed.sort(key=lambda x: x[0], reverse=True)

    def decode_tokens(tokens):
        out = []
        for idx in tokens:
            ch = seq2seq_tok.idx2char.get(idx, seq2seq_tok.UNK)
            if ch == seq2seq_tok.EOS:
                break
            if ch not in (seq2seq_tok.PAD, seq2seq_tok.SOS):
                out.append(ch)
        return "".join(out)

    seen, results = set(), []
    for lp, tokens in completed:
        word = decode_tokens(tokens)
        if word and word not in seen:
            seen.add(word)
            norm_score = math.exp(lp / max(len(tokens), 1))
            results.append((word, round(norm_score, 4)))
        if len(results) == 3:
            break

    return results


print("✓ Inference functions ready")

✓ Inference functions ready


## Step 5: Test inference

In [31]:
# ============================================================================
# TEST WORD DETECTION & CORRECTION
# ============================================================================

print("\n" + "="*70)
print("WORD DETECTION & CORRECTION TEST")
print("="*70 + "\n")

test_words = [
    ("यसरी", True),
    ("ीसरय", False),
    ("नेपालमा", True),
    ("नपेामला", False),
    # house in nepali 
    ("घर", True),
    ("घरम", True),
    ("hello", True),
    ("helo", False),
]

for word, expected_correct in test_words:
    is_correct, conf = detect_word(word, threshold=0.5)
    status = "✓" if is_correct else "✗"
    
    print(f"{status} {word}")
    print(f"   Status: {'CORRECT' if is_correct else 'INCORRECT'} (conf: {conf})")
    
    if not is_correct:
        suggestions = correct_word_beam(word, beam_width=5, max_len=MAX_WORD_LEN)
        if suggestions:
            print(f"   Suggestions:")
            for i, (sugg, score) in enumerate(suggestions, 1):
                print(f"     {i}. {sugg} ({score})")
    print()


WORD DETECTION & CORRECTION TEST

✗ यसरी
   Status: INCORRECT (conf: 0.2592)
   Suggestions:
     1. यसरी (0.9978)
     2. ससरी (0.2958)
     3. यससी (0.2857)

✓ ीसरय
   Status: CORRECT (conf: 1.0)

✗ नेपालमा
   Status: INCORRECT (conf: 0.0233)
   Suggestions:
     1. नेपालमा (0.998)
     2. नेपालमे (0.4577)
     3. नेपालले (0.446)

✓ नपेामला
   Status: CORRECT (conf: 1.0)

✗ घर
   Status: INCORRECT (conf: 0.0159)
   Suggestions:
     1. रघ (0.937)
     2. घघ (0.5546)
     3. घर (0.1813)

✗ घरम
   Status: INCORRECT (conf: 0.0345)
   Suggestions:
     1. घरम (0.8033)
     2. घरर (0.7523)
     3. घमर (0.6858)

✓ hello
   Status: CORRECT (conf: 0.7711)

✓ helo
   Status: CORRECT (conf: 0.6113)



In [38]:
# ============================================================================
# SENTENCE CORRECTION
# ============================================================================

def correct_sentence(text, threshold=0.5, beam_width=5):
    """
    Correct all words in a sentence.
    """
    words = text.split()
    corrected = []
    details = []
    
    for word in words:
        is_correct, conf = detect_word(word, threshold=threshold)
        
        if is_correct:
            corrected.append(word)
            details.append((word, "correct", conf, []))
        else:
            suggestions = correct_word_beam(word, beam_width=beam_width, max_len=MAX_WORD_LEN)
            best = suggestions[0][0] if suggestions else word
            corrected.append(best)
            details.append((word, "corrected", conf, suggestions))
    
    return {
        "input": text,
        "output": " ".join(corrected),
        "details": details
    }


print("\n" + "="*70)
print("SENTENCE CORRECTION")
print("="*70 + "\n")

test_sentences = [
     # 'वस्त' is incorrect
    "ल्दीिल",  # All words are incorrect
    # Both are incorrect
]

for sent in test_sentences:
    result = correct_sentence(sent, threshold=0.2, beam_width=5)
    print(f"Input:  {result['input']}")
    print(f"Output: {result['output']}")
    for word, status, conf, sugg in result['details']:
        if status == "corrected":
            tops = " | ".join([f"{w}({s})" for w, s in sugg[:2]])
            print(f"  └─ {word} → [{tops}]")
    print()


SENTENCE CORRECTION

Input:  ल्दीिल
Output: ल्दीिल

